Import

In [3]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

Loading

In [4]:
all_Q5107 = pd.read_csv(
    "../data/Q5107 All National General.csv",
    low_memory=False,
    dtype_backend="numpy_nullable"
    )
print(all_Q5107.head)
print(len(all_Q5107)) # Should be 317874

<bound method NDFrame.head of         PROVIDER_ID                               PROVIDER_NAME  \
0              5334                         Medical City Denton   
1              5334                         Medical City Denton   
2              5334                         Medical City Denton   
3              5334                         Medical City Denton   
4              5334                         Medical City Denton   
...             ...                                         ...   
317869        10148  Mercy Heart and Vascular Hospital St Louis   
317870        10148  Mercy Heart and Vascular Hospital St Louis   
317871        10148  Mercy Heart and Vascular Hospital St Louis   
317872        10148  Mercy Heart and Vascular Hospital St Louis   
317873        10148  Mercy Heart and Vascular Hospital St Louis   

       MEDICARE_PROVIDER_ID         NPI        EIN  \
0                    450634  1306897277  621682213   
1                    450634  1306897277  621682213   
2  

Drop Outliers

In [5]:
317874-138362

179512

In [6]:
print(all_Q5107["RATE_IS_OUTLIER"].sum())
all_Q5107_cleaned = all_Q5107[all_Q5107["RATE_IS_OUTLIER"]==False] # Remove outliers
print(len(all_Q5107_cleaned)) # Should be 179512
print(all_Q5107_cleaned.head)
print(len(all_Q5107_cleaned))
print("CLEANED: NA per column:")
print(all_Q5107_cleaned.isna().sum()) # List NA for each column
print("CLEANED: Number unique per column:")
print(all_Q5107_cleaned.nunique())

138362
179512
<bound method NDFrame.head of         PROVIDER_ID                               PROVIDER_NAME  \
1              5334                         Medical City Denton   
2              5334                         Medical City Denton   
3              5334                         Medical City Denton   
4              5334                         Medical City Denton   
5              5334                         Medical City Denton   
...             ...                                         ...   
317861        10148  Mercy Heart and Vascular Hospital St Louis   
317862        10148  Mercy Heart and Vascular Hospital St Louis   
317863        10148  Mercy Heart and Vascular Hospital St Louis   
317871        10148  Mercy Heart and Vascular Hospital St Louis   
317873        10148  Mercy Heart and Vascular Hospital St Louis   

       MEDICARE_PROVIDER_ID         NPI        EIN  \
1                    450634  1306897277  621682213   
2                    450634  1306897277  62

Check Ingest Dates

In [7]:
print(all_Q5107_cleaned["LOADED_ON"].value_counts())
print(all_Q5107_cleaned["INGESTED_ON"].value_counts())

LOADED_ON
2026-06-02 20:57:43.740    100537
2026-06-10 05:59:09.756     36210
2026-06-09 05:47:45.893     17949
2026-06-10 10:24:26.355      9666
2026-06-15 05:56:06.716      5106
2026-06-11 21:33:16.661      4387
2026-06-18 01:14:19.529      2031
2026-06-08 03:53:01.778      1855
2026-06-15 20:46:53.553      1671
2026-06-16 16:56:17.553        46
2026-06-04 20:34:18.863        29
2026-06-05 21:34:39.107        25
Name: count, dtype: Int64
INGESTED_ON
2026-05-04 15:08:38.108    1627
2026-05-26 20:18:59.990    1627
2026-05-02 01:17:20.088    1627
2026-05-04 14:30:19.035    1627
2026-05-04 17:33:57.644    1627
                           ... 
2026-04-20 01:38:09.968       1
2026-04-20 01:21:24.388       1
2026-04-13 01:08:20.772       1
2026-05-26 14:35:11.710       1
2026-03-16 01:38:18.606       1
Name: count, Length: 3063, dtype: Int64


Find Providers That Share Location With Different ID

Findings: some parts of the same campus are different "providers" (example: main vs childrens' hospital)

In [8]:
same_address_diff_ID = all_Q5107_cleaned[all_Q5107_cleaned.groupby("STREET_ADDRESS")["PROVIDER_ID"].transform("nunique")>1]

Check For Duplicates

In [28]:
print("Are there duplicates?")
print(all_Q5107_cleaned.duplicated(subset = ["PROVIDER_ID", "PAYER_ID", "PLAN_NAME", "NPI", "ZIP_CODE"]).any())

duplicates = all_Q5107_cleaned[all_Q5107_cleaned.duplicated(subset = ["PROVIDER_ID", "PAYER_ID", "PLAN_NAME"], keep = False)]
print(len(duplicates))
duplicates_entire_row = all_Q5107_cleaned[all_Q5107_cleaned.duplicated(keep = False)]
print(len(duplicates_entire_row))




Are there duplicates?
True
150555
59554


Check Missing

In [9]:
print(all_Q5107_cleaned[["NEGOTIATED_DOLLAR","NEGOTIATED_PERCENTAGE","GROSS_CHARGE", "ESTIMATED_ALLOWED_AMOUNT"]].isna().all(axis=1).sum())
# Should be 978

978


Drop Provider-Payer Rates Missing All Rates

In [10]:
all_Q5107_cleaned_HAS_RATE = all_Q5107_cleaned[~all_Q5107_cleaned[["NEGOTIATED_DOLLAR","NEGOTIATED_PERCENTAGE","GROSS_CHARGE", "ESTIMATED_ALLOWED_AMOUNT"]].isna().all(axis=1)]

len(all_Q5107_cleaned_HAS_RATE)

178534

Calculate Columns

In [11]:
all_Q5107_cleaned_HAS_RATE["NEGOTIATED_PERCENTAGE_TO_RATE"] = (
    all_Q5107_cleaned_HAS_RATE["NEGOTIATED_PERCENTAGE"] * all_Q5107_cleaned_HAS_RATE["GROSS_CHARGE"] * 0.01
)

# Default to NEGOTIATED_DOLLAR, then NEGOTIATED_PERCENTAGE_TO_RATE, then ESTIMATED_ALLOWED_AMOUNT
all_Q5107_cleaned_HAS_RATE["BEST_RATE"] = (
    all_Q5107_cleaned_HAS_RATE[["NEGOTIATED_DOLLAR", "NEGOTIATED_PERCENTAGE_TO_RATE", "ESTIMATED_ALLOWED_AMOUNT",]]
    .bfill(axis=1)
    .iloc[:, 0]
)

Create Seperate Table For All Providers

In [12]:
print(all_Q5107_cleaned_HAS_RATE[["NEGOTIATED_DOLLAR", "NEGOTIATED_PERCENTAGE"]].notna().all(axis=1).sum())
print(all_Q5107_cleaned_HAS_RATE[["NEGOTIATED_DOLLAR", "NEGOTIATED_PERCENTAGE", "ESTIMATED_ALLOWED_AMOUNT"]].notna().all(axis=1).sum())

12879
60


In [13]:
providers = (
    all_Q5107_cleaned_HAS_RATE
    .groupby("PROVIDER_ID", as_index = "False")
    .agg(
        PROVIDER_NAME = ("PROVIDER_NAME", "first"),
        HEALTH_SYSTEM_NAME = ("HEALTH_SYSTEM_NAME", "first"),
        STATE = ("STATE", "first"),
        STREET_ADDRESS = ("STREET_ADDRESS", "first"),
        GROSS_CHARGE = ("GROSS_CHARGE", "first"), # TODO: Check if different
        MEDIAN_DISCOUNTED_CASH_RATE = ("DISCOUNTED_CASH_RATE", "median"),
        MEDIAN_NEGOTIATED_DOLLAR = ("NEGOTIATED_DOLLAR", "median"),
        MEDIAN_NEGOTIATED_PERCENTAGE = ("NEGOTIATED_PERCENTAGE", "median"),
        MEDIAN_NEGOTIATED_PERCENTAGE_TO_RATE = ("NEGOTIATED_PERCENTAGE_TO_RATE", "median"),
        MEDIAN_ESTIMATED_ALLOWED_AMOUNT = ("ESTIMATED_ALLOWED_AMOUNT", "median"),
        MEDIAN_BEST_RATE = ("BEST_RATE", "median")

    )
)

print(providers[["MEDIAN_DISCOUNTED_CASH_RATE","MEDIAN_NEGOTIATED_DOLLAR","MEDIAN_NEGOTIATED_PERCENTAGE_TO_RATE", "MEDIAN_ESTIMATED_ALLOWED_AMOUNT", "GROSS_CHARGE"]].isna().all(axis=1).sum()) # TODO: why >0?, probably because has one of negotiated percentage and gross charge but not other

41


In [14]:
print(len(providers)) # Should be 3063 --> to 3057 after dropping

3057


Calculate Medians per Provider